# 🔮 Mini-Projet : Prédiction du Churn Client
## Systèmes Intelligents — Telco Customer Churn

**Objectif** : Concevoir un système intelligent de prédiction du churn client de bout en bout.

**Pipeline** : Exploration → Prétraitement → Modélisation Supervisée → Analyse Non Supervisée → Interface

---

## 1️⃣ Exploration des Données (EDA)

### 1.1 Chargement et aperçu du dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Configuration graphique
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

# Chargement
df = pd.read_csv('../data/telco_churn.csv')
print(f'Dimensions : {df.shape[0]} lignes × {df.shape[1]} colonnes')
df.head()

### 1.2 Informations générales

In [ ]:
print('Types de données :')
print(df.dtypes)
print(f'\nVariables numériques : {len(df.select_dtypes(include=[np.number]).columns)}')
print(f'Variables catégorielles : {len(df.select_dtypes(include="object").columns)}')

In [ ]:
df.describe()

In [ ]:
df.describe(include='object')

### 1.3 Valeurs manquantes

In [ ]:
# Valeurs manquantes classiques
missing = df.isnull().sum()
print('Valeurs manquantes (NaN) :', missing.sum())

# TotalCharges contient des espaces vides (pas des NaN)
empty_total = df[df['TotalCharges'] == ' ']
print(f'Valeurs vides dans TotalCharges : {len(empty_total)}')
print(f'\nCes {len(empty_total)} lignes correspondent à des nouveaux clients (tenure=0) :')
empty_total[['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges']].head()

### 1.4 Distribution de la variable cible (Churn)

In [ ]:
churn_counts = df['Churn'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colors = ['#2ecc71', '#e74c3c']

# Barplot
axes[0].bar(['Non Churn', 'Churn'], churn_counts.values, color=colors, edgecolor='white', linewidth=2)
axes[0].set_title('Distribution du Churn', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Nombre de clients')
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 50, str(v), ha='center', fontweight='bold', fontsize=12)

# Pie chart
axes[1].pie(churn_counts.values, labels=['Non Churn', 'Churn'],
            autopct='%1.1f%%', colors=colors, startangle=90,
            explode=(0, 0.05), shadow=True, textprops={'fontsize': 12})
axes[1].set_title('Proportion du Churn', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print(f'Le dataset est DESEQUILIBRE : {churn_counts.values[0]/len(df)*100:.1f}% Non Churn vs {churn_counts.values[1]/len(df)*100:.1f}% Churn')

### 1.5 Distribution des variables numériques

In [ ]:
df_temp = df.copy()
df_temp['TotalCharges'] = pd.to_numeric(df_temp['TotalCharges'], errors='coerce')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, col in enumerate(['tenure', 'MonthlyCharges', 'TotalCharges']):
    sns.histplot(data=df_temp, x=col, hue='Churn', kde=True, ax=axes[i],
                 palette=['#2ecc71', '#e74c3c'], alpha=0.6)
    axes[i].set_title(f'Distribution de {col}', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

### 1.6 Taux de churn par variable catégorielle

In [ ]:
cat_cols = ['gender', 'SeniorCitizen', 'Partner', 'Dependents',
            'PhoneService', 'InternetService', 'Contract',
            'PaymentMethod', 'PaperlessBilling']

fig, axes = plt.subplots(3, 3, figsize=(18, 15))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    churn_rate = df.groupby(col)['Churn'].apply(lambda x: (x == 'Yes').mean())
    churn_rate.plot(kind='bar', ax=axes[i], color=sns.color_palette('husl', len(churn_rate)), edgecolor='white')
    axes[i].set_title(f'Taux de Churn par {col}', fontsize=11, fontweight='bold')
    axes[i].set_ylabel('Taux de Churn')
    axes[i].tick_params(axis='x', rotation=45)
    axes[i].set_ylim(0, 0.7)
    axes[i].axhline(y=df['Churn'].apply(lambda x: 1 if x == 'Yes' else 0).mean(),
                     color='red', linestyle='--', alpha=0.5)

plt.suptitle('Taux de Churn par Variable Catégorielle', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

### 1.7 Matrice de corrélation

In [ ]:
df_corr = df_temp.copy()
df_corr['Churn'] = df_corr['Churn'].map({'Yes': 1, 'No': 0})
numeric_df = df_corr.select_dtypes(include=[np.number])

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(numeric_df.corr(), dtype=bool))
sns.heatmap(numeric_df.corr(), mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5, ax=ax)
ax.set_title('Matrice de Corrélation', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

### 📋 Résumé EDA

| Observation | Détail |
|-------------|--------|
| Dataset déséquilibré | ~73% Non Churn vs ~27% Churn |
| Contrat mensuel | Taux de churn très élevé |
| Nouveaux clients | Churnent davantage (faible tenure) |
| Charges élevées | Corrélées au churn |
| Electronic check | Méthode de paiement associée au churn |
| Services absents | Pas de sécurité/support → plus de churn |

---

## 2️⃣ Prétraitement des Données

### 2.1 Nettoyage

In [ ]:
import sys
sys.path.append('..')
from src.preprocessing import load_data, clean_data, encode_features, feature_engineering, scale_features, prepare_data
from sklearn.model_selection import train_test_split

df_raw = load_data('../data/telco_churn.csv')
print(f'Avant nettoyage : {df_raw.shape}')
print(f'TotalCharges dtype : {df_raw["TotalCharges"].dtype}')

df_clean = clean_data(df_raw)
print(f'\nAprès nettoyage : {df_clean.shape}')
print(f'TotalCharges dtype : {df_clean["TotalCharges"].dtype}')
print(f'Valeurs manquantes : {df_clean.isnull().sum().sum()}')
print(f'customerID supprimé : {"customerID" not in df_clean.columns}')

### 2.2 Feature Engineering

In [ ]:
df_feat = feature_engineering(df_clean)

print('Nouvelles features créées :')
print('  - ChargesPerTenure : MonthlyCharges / (tenure + 1)')
print('  - IsNewCustomer   : 1 si tenure <= 6 mois')
df_feat[['tenure', 'MonthlyCharges', 'ChargesPerTenure', 'IsNewCustomer']].head(10)

### 2.3 Encodage des variables catégorielles

In [ ]:
cat_before = df_feat.select_dtypes(include='object').columns.tolist()
print(f'Variables catégorielles avant encodage ({len(cat_before)}) : {cat_before}')

df_encoded = encode_features(df_feat)
print(f'\nDimensions après encodage : {df_encoded.shape}')
print(f'Toutes numériques : {df_encoded.select_dtypes(include="object").shape[1] == 0}')
print(f'\nColonnes ({len(df_encoded.columns)}) : {list(df_encoded.columns)}')

### 2.4 Split Train/Test et Normalisation

In [ ]:
X = df_encoded.drop('Churn', axis=1)
y = df_encoded['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f'Train : {X_train.shape[0]} ({X_train.shape[0]/len(X)*100:.0f}%)')
print(f'Test  : {X_test.shape[0]} ({X_test.shape[0]/len(X)*100:.0f}%)')
print(f'\nDistribution cible (train) :\n{y_train.value_counts(normalize=True)}')

# Normalisation
X_train_s, X_test_s, scaler = scale_features(X_train, X_test)
print(f'\nMoyenne après normalisation : ~{X_train_s.mean().mean():.4f}')
print(f'Ecart-type après normalisation : ~{X_train_s.std().mean():.4f}')

---

## 3️⃣ Modélisation Supervisée

### 3.1 Entraînement de 5 modèles

In [ ]:
from src.models import compare_models, cross_validate_models, save_best_model
from src.preprocessing import prepare_data
import joblib

# Pipeline complet
X_train, X_test, y_train, y_test, scaler, df_clean, feature_names = prepare_data('../data/telco_churn.csv')

print(f'Train : {X_train.shape}, Test : {X_test.shape}')
print(f'Features : {len(feature_names)}')

In [ ]:
# Entraînement et comparaison
comparison_df, all_results = compare_models(X_train, X_test, y_train, y_test)
print('\n' + '='*60)
print('TABLEAU DE COMPARAISON')
print('='*60)
comparison_df

### 3.2 Graphique de comparaison

In [ ]:
# Comparaison visuelle
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(comparison_df))
width = 0.18
colors_m = ['#3498db', '#2ecc71', '#e67e22', '#9b59b6']

for i, metric in enumerate(metrics):
    values = comparison_df[metric].astype(float)
    ax.bar(x + i*width, values, width, label=metric, color=colors_m[i], edgecolor='white')

ax.set_ylabel('Score')
ax.set_title('Comparaison des Modèles', fontsize=16, fontweight='bold')
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(comparison_df['Modèle'], rotation=15, ha='right')
ax.legend()
ax.set_ylim(0, 1.1)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### 3.3 Matrices de confusion

In [ ]:
n = len(all_results)
fig, axes = plt.subplots(1, n, figsize=(5*n, 4))

for i, (name, res) in enumerate(all_results.items()):
    sns.heatmap(res['Confusion Matrix'], annot=True, fmt='d', cmap='Blues', ax=axes[i],
                xticklabels=['Non Churn', 'Churn'], yticklabels=['Non Churn', 'Churn'])
    axes[i].set_title(name, fontsize=12, fontweight='bold')
    axes[i].set_xlabel('Prédit'); axes[i].set_ylabel('Réel')

plt.suptitle('Matrices de Confusion', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 3.4 Courbes ROC

In [ ]:
from sklearn.metrics import roc_curve, auc

fig, ax = plt.subplots(figsize=(10, 8))
colors_roc = sns.color_palette('husl', len(all_results))

for i, (name, res) in enumerate(all_results.items()):
    if res['y_prob'] is not None:
        fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, color=colors_roc[i], lw=2, label=f'{name} (AUC={roc_auc:.3f})')

ax.plot([0,1], [0,1], 'k--', lw=1, label='Baseline')
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title('Courbes ROC', fontsize=14, fontweight='bold')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 3.5 Importance des features (Random Forest)

In [ ]:
rf = all_results['Random Forest']['model']
importances = rf.feature_importances_
indices = np.argsort(importances)[::-1][:15]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(range(15), importances[indices][::-1], color=sns.color_palette('viridis', 15))
ax.set_yticks(range(15))
ax.set_yticklabels([feature_names[i] for i in indices][::-1])
ax.set_xlabel('Importance')
ax.set_title('Top 15 Features les Plus Importantes', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 3.6 Validation croisée (5-Fold)

In [ ]:
print('Validation Croisée (5-Fold) :')
cv_results = cross_validate_models(X_train, y_train, cv=5)

fig, ax = plt.subplots(figsize=(10, 6))
names = list(cv_results.keys())
means = [cv_results[n]['Mean F1'] for n in names]
stds = [cv_results[n]['Std F1'] for n in names]

bars = ax.bar(names, means, yerr=stds, capsize=5, color=sns.color_palette('husl', len(names)), edgecolor='white')
ax.set_ylabel('F1-Score')
ax.set_title('Validation Croisée (5-Fold)', fontsize=14, fontweight='bold')
ax.set_ylim(0, 1)
for bar, mean in zip(bars, means):
    ax.text(bar.get_x()+bar.get_width()/2., bar.get_height()+0.02, f'{mean:.3f}', ha='center', fontweight='bold')
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.show()

### 3.7 Sauvegarde du meilleur modèle

In [ ]:
best_name, best_model = save_best_model(all_results, '../models/best_model.pkl')
joblib.dump(scaler, '../models/scaler.pkl')
joblib.dump(feature_names, '../models/feature_names.pkl')
print('\nFichiers sauvegardés :')
print('  ✅ models/best_model.pkl')
print('  ✅ models/scaler.pkl')
print('  ✅ models/feature_names.pkl')

---

## 4️⃣ Analyse Non Supervisée

### 4.1 K-Means : Méthode du coude

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

X_all = pd.concat([X_train, X_test])
y_all = pd.concat([y_train, y_test])

inertias, silhouettes = [], []
K_range = range(2, 10)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_all)
    inertias.append(km.inertia_)
    sil = silhouette_score(X_all, km.labels_)
    silhouettes.append(sil)
    print(f'  k={k} : Silhouette={sil:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('k'); axes[0].set_ylabel('Inertie')
axes[0].set_title('Méthode du Coude', fontsize=14, fontweight='bold')

axes[1].plot(K_range, silhouettes, 'ro-', linewidth=2, markersize=8)
axes[1].set_xlabel('k'); axes[1].set_ylabel('Silhouette')
axes[1].set_title('Score de Silhouette', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 4.2 Clustering avec k=3

In [ ]:
best_k = 3
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_all)

print(f'Silhouette Score : {silhouette_score(X_all, clusters):.4f}')
print(f'\nDistribution des clusters :')
for i in range(best_k):
    count = (clusters == i).sum()
    print(f'  Cluster {i} : {count} clients ({count/len(clusters)*100:.1f}%)')

### 4.3 PCA — Réduction de dimensionnalité

In [ ]:
pca_full = PCA()
pca_full.fit(X_all)
cumulative = np.cumsum(pca_full.explained_variance_ratio_)
n_90 = np.argmax(cumulative >= 0.90) + 1

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(1, len(cumulative)+1), cumulative, 'ro-', linewidth=2)
ax.axhline(y=0.90, color='green', linestyle='--', label='90% variance')
ax.axhline(y=0.95, color='orange', linestyle='--', label='95% variance')
ax.set_xlabel('Nombre de composantes')
ax.set_ylabel('Variance cumulative')
ax.set_title('Variance Cumulative (PCA)', fontsize=14, fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print(f'{n_90} composantes nécessaires pour 90% de la variance')

### 4.4 Visualisation 2D (PCA + Clusters)

In [ ]:
pca_2d = PCA(n_components=2)
X_pca = pca_2d.fit_transform(X_all)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sc1 = axes[0].scatter(X_pca[:,0], X_pca[:,1], c=clusters, cmap='viridis', alpha=0.5, s=10)
axes[0].set_title('Clusters K-Means (PCA 2D)', fontsize=14, fontweight='bold')
axes[0].set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}%)')
axes[0].set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}%)')
plt.colorbar(sc1, ax=axes[0], label='Cluster')

sc2 = axes[1].scatter(X_pca[:,0], X_pca[:,1], c=y_all.values, cmap='RdYlGn_r', alpha=0.5, s=10)
axes[1].set_title('Churn Réel (PCA 2D)', fontsize=14, fontweight='bold')
axes[1].set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}%)')
axes[1].set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}%)')
plt.colorbar(sc2, ax=axes[1], label='Churn')

plt.tight_layout()
plt.show()

### 4.5 Profiling des segments & Taux de churn

In [ ]:
df_analysis = X_all.copy()
df_analysis['Cluster'] = clusters
df_analysis['Churn'] = y_all.values

# Taux de churn par cluster
churn_by_cluster = df_analysis.groupby('Cluster')['Churn'].mean()

fig, ax = plt.subplots(figsize=(8, 5))
colors_c = ['#2ecc71', '#f39c12', '#e74c3c']
bars = ax.bar(range(best_k), churn_by_cluster.values, color=colors_c, edgecolor='white', linewidth=2)
ax.set_xlabel('Cluster'); ax.set_ylabel('Taux de Churn')
ax.set_title('Taux de Churn par Segment Client', fontsize=14, fontweight='bold')
ax.set_xticks(range(best_k))
ax.set_xticklabels([f'Segment {i}' for i in range(best_k)])
for bar, rate in zip(bars, churn_by_cluster.values):
    ax.text(bar.get_x()+bar.get_width()/2., bar.get_height()+0.01, f'{rate*100:.1f}%', ha='center', fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# Profil moyen
imp_feats = [f for f in ['tenure','MonthlyCharges','TotalCharges','ChargesPerTenure','IsNewCustomer'] if f in df_analysis.columns]
print('\nProfil moyen par segment :')
df_analysis.groupby('Cluster')[imp_feats].mean().round(3)

---

## 5️⃣ Conclusion

### Résultats principaux
- **5 modèles** entraînés et comparés (Logistic Regression, Random Forest, SVM, k-NN, Decision Tree)
- **Meilleur modèle** sélectionné automatiquement selon le F1-Score
- **3 segments** de clients identifiés par K-Means avec des taux de churn différenciés
- **PCA** confirme la structure des données et la séparabilité partielle

### Limites
- Dataset déséquilibré (pourrait bénéficier de SMOTE)
- Pas d'optimisation des hyperparamètres (GridSearch)
- Features limitées au dataset disponible

### Pistes d'amélioration
- Appliquer SMOTE pour rééquilibrer les classes
- GridSearchCV pour optimiser les hyperparamètres
- Tester des modèles avancés (XGBoost, LightGBM)
- Ajouter des features temporelles

### Interface
L'application Streamlit (`app/streamlit_app.py`) permet de prédire le churn en temps réel :
```bash
streamlit run app/streamlit_app.py
```